## 1. Imports & Setup

The cleaning pipeline from notebook 02 is replayed here in one compact block so this notebook is self-contained and reproducible without manual state transfer.

In [1]:
import numpy as np
import pandas as pd

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("../data/mercari_sample.csv")

# ── Cleaning (mirrors 02_cleaning.ipynb) ──────────────────────────────────────
df["brand_name"] = df["brand_name"].fillna("No Brand")   # 43.2% null → sentinel
df = df.dropna(subset=["category_name"])                  # 0.47% null → drop
df = df[df["price"] > 0]                                  # free items → drop
df = df.drop_duplicates()                                  # defensive; 0 found in 02

# Normalise brand casing for consistent groupby keys
df["brand_name"] = df["brand_name"].str.lower().str.strip()

print(f"Clean DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
assert df.isnull().sum().sum() == 0, "Unexpected nulls — check cleaning pipeline"
print("✓ Zero nulls confirmed")

Clean DataFrame: 49,725 rows × 8 columns
✓ Zero nulls confirmed


## 2. Feature Engineering

Features are built in dependency order: category splits first (downstream features reference `main_category`), then text, binary, price transforms, and finally aggregations that consume multiple earlier columns.

### 2.1 Category Hierarchy

`category_name` encodes up to three levels separated by `/` (e.g. `Women/Tops & Blouses/Blouse`).  
Splitting into separate columns lets each level be used independently in groupby, pivot, and model features.

In [2]:
# Parse once into a Series of lists, then index each level — one split, zero re-computation
cat_split = df["category_name"].str.split("/")

df["main_category"]    = cat_split.str[0]   # e.g. "Women"          — primary price driver
df["sub_category"]     = cat_split.str[1]   # e.g. "Tops & Blouses" — secondary signal
df["sub_sub_category"] = cat_split.str[2]   # e.g. "Blouse"         — most granular signal

### 2.2 Text Signals

Title and description length are proxies for seller effort — a longer, more detailed listing correlates with higher perceived value and often higher price.

In [3]:
# Character counts capture information density regardless of word choice
df["name_length"] = df["name"].str.len()
df["desc_length"] = df["item_description"].str.len()

# Word count differs from character count in signal: a 5-word title uses different
# vocabulary than a 20-word title and often signals item specificity
df["name_word_count"] = df["name"].str.split().str.len()

### 2.3 Binary Indicators

Two high-signal binary flags: whether the seller bothered to write a description (listing quality), and whether the item carries a brand (premium capture).

In [4]:
# Mercari fills undescribed items with the literal string "No description yet"
# Converting to binary avoids treating that string as meaningful text downstream
df["has_description"] = (
    df["item_description"].ne("No description yet").astype(int)
)

# brand_name was filled with "no brand" (lowercased) during cleaning —
# that sentinel is the only value that maps to 0 here
df["is_branded"] = np.where(df["brand_name"] == "no brand", 0, 1)

### 2.4 Condition Label

`item_condition_id` is an ordinal integer (1–5). Mapping it to human-readable labels makes groupby outputs and chart axes self-explanatory without losing ordinality.

In [5]:
condition_map = {1: "New", 2: "Like New", 3: "Good", 4: "Fair", 5: "Poor"}
df["condition_label"] = df["item_condition_id"].map(condition_map)

### 2.5 Price Transforms

Raw `price` is right-skewed (mean \$26.65, max \$1,506). Three derived columns normalize it for modelling and segment it for analysis:

- **`log_price`** — compresses the tail; standard target for price regression
- **`is_price_outlier`** — IQR fence flags extreme values without hard-coding a dollar threshold
- **`price_tier`** — coarse segment (budget / mid / premium) for group-level pattern discovery

In [6]:
# log1p(x) = log(1 + x): handles hypothetical zero-price edge cases gracefully
df["log_price"] = np.log1p(df["price"])

# IQR fence — robust to skew; avoids anchoring the outlier boundary to the mean
Q1 = np.percentile(df["price"], 25)
Q3 = np.percentile(df["price"], 75)
IQR = Q3 - Q1
df["is_price_outlier"] = np.where(
    (df["price"] < Q1 - 1.5 * IQR) | (df["price"] > Q3 + 1.5 * IQR), 1, 0
)

# Tier boundaries chosen from domain knowledge: sub-$10 is typical Mercari floor;
# >$50 skews toward electronics, luxury, and collectibles
df["price_tier"] = np.where(
    df["price"] <= 10, "budget",
    np.where(df["price"] <= 50, "mid", "premium")
)

### 2.6 Aggregation & Target Encoding

These four features embed group-level price statistics directly into each row — a lightweight form of target encoding that gives any downstream model immediate access to "what does this category / brand typically sell for" without requiring a join.

In [7]:
# Category-level mean price — primary group signal for price positioning
cat_avg = df.groupby("main_category")["price"].mean()
df["cat_avg_price"] = df["main_category"].map(cat_avg)

# Brand-level mean price — captures brand premium independent of category
brand_avg = df.groupby("brand_name")["price"].mean()
df["brand_avg_price"] = df["brand_name"].map(brand_avg)

# Rank within category — where does this item sit relative to its peers?
# dense ranking avoids gaps when tie-groups exist
df["price_rank_in_cat"] = (
    df.groupby("main_category")["price"]
      .rank(ascending=False, method="dense")
      .astype(int)
)

# Global price percentile — normalised [0, 1] position across the entire dataset
df["price_percentile"] = df["price"].rank(pct=True).round(3)

### 2.7 Category Depth

Not all listings are categorised to the same depth — some carry all three levels, others only one or two. Depth is a proxy for listing specificity: a seller who picks `Women/Tops & Blouses/Blouse` has put more effort into classification than one who stops at `Women`.

In [8]:
# Counts the number of "/"-separated segments: depth 1 = top-level only, depth 3 = fully specified
df["category_depth"] = df["category_name"].str.split("/").str.len()

## 3. Feature Inventory

All 17 engineered features — their type, origin column, and the price signal each encodes.

In [9]:
feature_docs = pd.DataFrame({
    "feature": [
        "main_category", "sub_category", "sub_sub_category",
        "name_length", "desc_length", "name_word_count",
        "has_description", "is_branded",
        "condition_label",
        "log_price", "is_price_outlier", "price_tier",
        "cat_avg_price", "brand_avg_price", "price_rank_in_cat", "price_percentile",
        "category_depth"
    ],
    "type": [
        "categorical", "categorical", "categorical",
        "numeric",     "numeric",     "numeric",
        "binary",      "binary",
        "categorical",
        "numeric",     "binary",       "categorical",
        "numeric",     "numeric",     "numeric",           "numeric",
        "numeric"
    ],
    "derived_from": [
        "category_name", "category_name", "category_name",
        "name",          "item_description", "name",
        "item_description", "brand_name",
        "item_condition_id",
        "price",         "price (IQR)",  "price (bins)",
        "groupby mean",  "groupby mean", "groupby rank",      "overall rank",
        "category_name"
    ],
    "signal": [
        "Primary price driver — Electronics vs Kids",
        "Secondary price signal within main category",
        "Most specific product-type signal",
        "Seller effort proxy via title length",
        "Seller effort proxy via description length",
        "Title richness — vocabulary breadth signal",
        "Listing quality flag",
        "Brand premium capture",
        "Human-readable condition for groupby / charts",
        "Normalised regression target (compresses right tail)",
        "Flags extreme pricing via IQR fence",
        "Coarse price segment for pattern discovery",
        "Target encoding — category-level price expectation",
        "Target encoding — brand-level price premium",
        "Relative price position within category peers",
        "Global price percentile across full dataset",
        "Listing specificity — depth of category selection"
    ]
})

feature_docs

,feature,type,derived_from,signal
0,main_category,categorical,category_name,Primary price driver — Electronics vs Kids
1,sub_category,categorical,category_name,Secondary price signal within main category
2,sub_sub_category,categorical,category_name,Most specific product-type signal
3,name_length,numeric,name,Seller effort proxy via title length
4,desc_length,numeric,item_description,Seller effort proxy via description length
5,name_word_count,numeric,name,Title richness — vocabulary breadth signal
6,has_description,binary,item_description,Listing quality flag
7,is_branded,binary,brand_name,Brand premium capture
8,condition_label,categorical,item_condition_id,Human-readable condition for groupby / charts
9,log_price,numeric,price,Normalised regression target (compresses right...


## 4. Feature Correlation with Price

Pearson correlation (absolute value) against raw `price` as a quick first sanity check on signal strength.  
Price-derived features (`log_price`, `is_price_outlier`, `price_percentile`) are expected to dominate — the actionable signals are those ranked below them.

In [10]:
numeric_features = [
    "price",
    "log_price", "is_price_outlier", "price_percentile",
    "brand_avg_price", "price_rank_in_cat", "is_branded",
    "cat_avg_price", "shipping", "desc_length",
    "category_depth", "name_word_count", "name_length",
    "has_description", "item_condition_id"
]

corr_with_price = (
    df[numeric_features]
      .corr()["price"]
      .drop("price")          # remove self-correlation
      .abs()
      .sort_values(ascending=False)
      .round(3)
      .rename("abs_corr_with_price")
)

corr_with_price.to_frame()

,abs_corr_with_price
log_price,0.752
is_price_outlier,0.673
price_percentile,0.570
brand_avg_price,0.499
price_rank_in_cat,0.263
is_branded,0.138
cat_avg_price,0.135
shipping,0.100
desc_length,0.048
category_depth,0.038


## 5. Final Shape Check

Confirm the DataFrame exits this notebook with the expected 17 new columns appended and zero nulls introduced.

In [11]:
new_features = list(feature_docs["feature"])

assert all(f in df.columns for f in new_features), "One or more features missing from DataFrame"
assert df[new_features].isnull().sum().sum() == 0, "Nulls introduced during feature engineering"

print("✓ All 17 features present")
print(f"  Final shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  New columns : {df.shape[1] - 8} (original 8 + 17 engineered)")

✓ All 17 features present
  Final shape : 49,725 rows × 25 columns
  New columns : 17 (original 8 + 17 engineered)
